# 🎓 বাংলা NCTB ডেটাসেট দিয়ে Qwen2.5-0.5B ফাইন-টিউনিং

এই নোটবুকটি দিয়ে আপনি **Qwen2.5-0.5B** মডেলকে বাংলা শিক্ষা ডেটাসেট দিয়ে ফাইন-টিউন করতে পারবেন।

**প্রয়োজনীয় সময়**: ৩০-৪০ মিনিট (Google Colab T4 GPU)

**ধাপসমূহ**:
1. পরিবেশ সেটআপ (৫ মিনিট)
2. ডেটাসেট লোড করা (২ মিনিট)
3. মডেল ফাইন-টিউনিং (২৫-৩৫ মিনিট)
4. টেস্টিং ও ডাউনলোড (৫ মিনিট)

---

## ⚠️ শুরু করার আগে:

1. **GPU চালু করুন**: Runtime → Change runtime type → Hardware accelerator: **T4 GPU**
2. **Google Drive মাউন্ট করুন**: পরবর্তী সেলে
3. **ডেটাসেট আপলোড করুন**: `nctb_dataset_500plus.jsonl` আপনার Drive-এ রাখুন

## 📌 ধাপ ১: GPU চেক করুন

In [ ]:
# GPU চেক করুন
!nvidia-smi

## 📁 ধাপ ২: Google Drive মাউন্ট করুন

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 📦 ধাপ ৩: প্রয়োজনীয় লাইব্রেরি ইনস্টল করুন

এই সেলটি চালাতে **৩-৫ মিনিট** সময় লাগবে।

In [ ]:
%%capture
# প্রয়োজনীয় প্যাকেজ ইনস্টল
!pip install -q transformers datasets accelerate peft bitsandbytes trl torch

## 🔧 ধাপ ৪: ডেটাসেট লোড ও প্রস্তুতি

**গুরুত্বপূর্ণ**: নিচের কোডে `DATASET_PATH` পরিবর্তন করে আপনার ডেটাসেটের সঠিক পাথ দিন।

In [ ]:
import json
from datasets import Dataset

# ✏️ আপনার ডেটাসেটের পাথ এখানে লিখুন
# নোট: এই রেপোতে ৪০৮টি ভ্যালিড NCTB প্রশ্ন-উত্তর আছে
DATASET_PATH = "/content/drive/MyDrive/nctb_dataset_500plus.jsonl"

# JSONL ফাইল লোড করুন (এরর হ্যান্ডলিং সহ)
data = []
error_lines = []

print("📂 ফাইল লোড হচ্ছে...\n")

with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:  # খালি লাইন স্কিপ
            continue
        
        try:
            # JSON parse করুন
            item = json.loads(line)
            
            # Validate: instruction এবং output আছে কিনা চেক করুন
            if 'instruction' in item and 'output' in item:
                data.append(item)
            else:
                print(f"⚠️  লাইন {line_num}: 'instruction' বা 'output' ফিল্ড নেই, স্কিপ করা হলো")
                error_lines.append(line_num)
        
        except json.JSONDecodeError as e:
            print(f"❌ লাইন {line_num}: JSON এরর - {str(e)[:50]}... স্কিপ করা হলো")
            error_lines.append(line_num)
            continue

print(f"\n{'='*60}")
print(f"✅ সফলভাবে লোড হয়েছে: {len(data)} টি")

if error_lines:
    print(f"⚠️  সমস্যাযুক্ত লাইন: {len(error_lines)} টি (লাইন নম্বর: {error_lines[:10]}{'...' if len(error_lines) > 10 else ''})")
else:
    print(f"🎉 সব ডেটা পারফেক্ট!")

print(f"{'='*60}\n")

# প্রথম কয়েকটি উদাহরণ দেখুন
if data:
    print(f"📝 প্রথম ৩টি উদাহরণ:\n")
    for i, example in enumerate(data[:3], 1):
        print(f"{i}. প্রশ্ন: {example['instruction'][:80]}...")
        print(f"   উত্তর: {example['output'][:80]}...\n")
else:
    print("❌ কোনো ভ্যালিড ডেটা লোড হয়নি! ফাইল চেক করুন।")
    raise ValueError("ডেটাসেট খালি!")

# Dataset অবজেক্ট তৈরি
dataset = Dataset.from_list(data)

# Train/Test split (95% train, 5% test)
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(f"\n📊 ডেটাসেট বিভাজন:")
print(f"  - Training: {len(dataset['train'])} টি")
print(f"  - Testing: {len(dataset['test'])} টি")
print(f"\n✅ ডেটাসেট প্রস্তুত!")

## 🤖 ধাপ ৫: Qwen2.5-0.5B মডেল ও টোকেনাইজার লোড করুন

এই সেলটি মডেল ডাউনলোড করবে (প্রথমবার ২-৩ মিনিট লাগতে পারে)।

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# মডেলের নাম
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# 4-bit quantization কনফিগ (মেমোরি সাশ্রয়ের জন্য)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("📥 মডেল লোড হচ্ছে...")

# টোকেনাইজার লোড
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# মডেল লোড (4-bit quantized)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# মডেলকে training-এর জন্য প্রস্তুত করুন
model = prepare_model_for_kbit_training(model)

print("✅ মডেল সফলভাবে লোড হয়েছে!")
print(f"\n🔢 মডেল প্যারামিটার: {MODEL_NAME}")
print(f"💾 মেমোরি ব্যবহার: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## ⚙️ ধাপ ৬: LoRA কনফিগারেশন সেটআপ

**LoRA** (Low-Rank Adaptation) ব্যবহার করে আমরা শুধু মডেলের একটি ছোট অংশ ট্রেন করব।

In [ ]:
# LoRA কনফিগ
lora_config = LoraConfig(
    r=16,                      # LoRA rank
    lora_alpha=32,             # LoRA alpha
    target_modules=[           # কোন লেয়ারে LoRA প্রয়োগ করবেন
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# মডেলে LoRA প্রয়োগ করুন
model = get_peft_model(model, lora_config)

# Trainable parameters দেখুন
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print(f"✅ LoRA কনফিগ সফল!")
print(f"\n📊 প্যারামিটার তথ্য:")
print(f"  - Trainable: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")
print(f"  - Total: {all_params:,}")

## 📝 ধাপ ৭: ডেটা ফরম্যাটিং

আমরা ডেটাকে Qwen-এর instruction format-এ রূপান্তর করব।

In [ ]:
def format_prompt(example):
    """
    NCTB ডেটাসেটকে Qwen instruction format-এ রূপান্তর করুন
    """
    # Qwen2.5 chat template
    text = f"""<|im_start|>system
You are a helpful AI assistant for Bengali education. Answer questions based on NCTB curriculum.<|im_end|>
<|im_start|>user
{example['instruction']}<|im_end|>
<|im_start|>assistant
{example['output']}<|im_end|>"""
    
    return {"text": text}

# ডেটাসেট ফরম্যাট করুন
formatted_dataset = dataset.map(format_prompt, remove_columns=dataset['train'].column_names)

print("✅ ডেটাসেট ফরম্যাট সম্পন্ন!")
print(f"\n📄 ফরম্যাটেড উদাহরণ (প্রথম ৩০০ অক্ষর):")
print(formatted_dataset['train'][0]['text'][:300] + "...")

## 🎯 ধাপ ৮: Training Arguments সেটআপ

এখানে আপনি training-এর বিভিন্ন প্যারামিটার কনফিগার করতে পারবেন।

In [ ]:
from transformers import TrainingArguments

# আউটপুট ডিরেক্টরি (Drive-এ সেভ হবে)
output_dir = "/content/drive/MyDrive/qwen_nctb_finetuned"

training_args = TrainingArguments(
    # আউটপুট
    output_dir=output_dir,
    
    # Training হাইপারপ্যারামিটার
    num_train_epochs=3,                    # ৩ ইপক (পুরো ডেটা ৩ বার দেখবে)
    per_device_train_batch_size=4,         # ব্যাচ সাইজ
    gradient_accumulation_steps=4,         # গ্রেডিয়েন্ট জমা
    
    # অপটিমাইজেশন
    learning_rate=2e-4,                    # লার্নিং রেট
    warmup_steps=50,                       # ওয়ার্মআপ স্টেপ
    weight_decay=0.01,                     # ওয়েট ডিকে
    max_grad_norm=1.0,                     # গ্রেডিয়েন্ট ক্লিপিং
    
    # মেমোরি সাশ্রয়
    fp16=False,                            # T4 এর জন্য bf16 ভালো
    bf16=True,                             # BFloat16 ব্যবহার করুন
    gradient_checkpointing=True,           # মেমোরি সাশ্রয়
    
    # Logging ও সেভ
    logging_steps=10,                      # প্রতি ১০ স্টেপে লগ
    save_steps=100,                        # প্রতি ১০০ স্টেপে সেভ
    save_total_limit=2,                    # সর্বোচ্চ ২টি চেকপয়েন্ট রাখুন
    
    # Evaluation
    eval_strategy="steps",                # মাঝে মাঝে evaluation
    eval_steps=100,                        # প্রতি ১০০ স্টেপে eval
    
    # অন্যান্য
    remove_unused_columns=False,
    report_to="none",                      # wandb বন্ধ
)

print("✅ Training Arguments সেটআপ সম্পন্ন!")
print(f"\n⏱️ আনুমানিক সময়:")
print(f"  - ডেটাসেট সাইজ: {len(dataset['train'])} টি")
print(f"  - Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  - প্রতি ইপকে স্টেপ: ~{len(dataset['train']) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
print(f"  - মোট স্টেপ: ~{(len(dataset['train']) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)) * training_args.num_train_epochs}")
print(f"  - সম্ভাব্য সময়: ২৫-৪০ মিনিট")

## 🚀 ধাপ ৯: Training শুরু করুন!

**এই সেলটি চালালে ট্রেনিং শুরু হবে। এটি ২৫-৪০ মিনিট সময় নিবে।**

আপনি প্রগ্রেস বার দেখে বুঝতে পারবেন কতটা হয়েছে। চা-কফি খেয়ে আসতে পারেন! ☕

In [ ]:
from trl import SFTTrainer

print("✅ Trainer imports successful!")

# Trainer তৈরি করুন (compatible with TRL >= 0.7.0)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset['train'],
    eval_dataset=formatted_dataset['test'],
    dataset_text_field="text",  # Field name containing the text
    packing=False,  # Don't pack multiple examples together
)

print("🚀 ট্রেনিং শুরু হচ্ছে...\n")
print("⏰ এখন আপনি চা-কফি খেতে পারেন। ২৫-৪০ মিনিট পর ফিরে আসুন!\n")
print("="*60)

# Training শুরু!
trainer.train()

print("\n" + "="*60)
print("🎉 ট্রেনিং সম্পন্ন! মডেল সেভ হচ্ছে...")

## 💾 ধাপ ১০: মডেল সেভ করুন

In [ ]:
# Final model save
final_model_path = "/content/drive/MyDrive/qwen_nctb_final"

print(f"💾 মডেল সেভ হচ্ছে: {final_model_path}")

# LoRA adapter সেভ করুন
trainer.model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("✅ মডেল সফলভাবে সেভ হয়েছে!")
print(f"\n📁 সেভ লোকেশন:")
print(f"  {final_model_path}")
print(f"\n📝 ফাইলসমূহ:")
!ls -lh {final_model_path}

## 🧪 ধাপ ১১: মডেল টেস্ট করুন

এখন আপনার ট্রেন করা মডেল দিয়ে কিছু প্রশ্নের উত্তর দেখুন!

In [ ]:
# টেস্ট করার জন্য মডেল লোড
from peft import PeftModel

print("🔄 মডেল লোড হচ্ছে টেস্টিং-এর জন্য...")

# Base model load
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# LoRA adapter merge
model_test = PeftModel.from_pretrained(base_model, final_model_path)
model_test = model_test.merge_and_unload()

print("✅ মডেল প্রস্তুত!\n")

# Test function
def test_model(question):
    prompt = f"""<|im_start|>system
You are a helpful AI assistant for Bengali education. Answer questions based on NCTB curriculum.<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model_test.device)
    
    outputs = model_test.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract only assistant's response
    try:
        answer = response.split("<|im_start|>assistant\n")[1].split("<|im_end|>")[0].strip()
    except:
        answer = response
    
    return answer

print("🧪 টেস্ট শুরু!\n")
print("="*60)

In [ ]:
# টেস্ট প্রশ্ন
test_questions = [
    "বাংলাদেশের রাজধানী কী?",
    "সালোকসংশ্লেষণ কাকে বলে?",
    "পিথাগোরাসের উপপাদ্য কী?",
    "বাংলা বর্ণমালায় মোট কয়টি বর্ণ আছে?",
    "What is the capital of Bangladesh?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n❓ প্রশ্ন {i}: {question}")
    answer = test_model(question)
    print(f"✅ উত্তর: {answer}")
    print("-"*60)

## 🎨 ধাপ ১২: নিজের প্রশ্ন করুন!

এখন আপনি নিজে কোনো প্রশ্ন করে দেখতে পারেন।

In [ ]:
# আপনার প্রশ্ন এখানে লিখুন
my_question = "মানবদেহে কয়টি হাড় আছে?"  # ✏️ এখানে আপনার প্রশ্ন লিখুন

print(f"❓ আপনার প্রশ্ন: {my_question}\n")
answer = test_model(my_question)
print(f"✅ মডেলের উত্তর:\n{answer}")

## 📤 ধাপ ১৩: মডেল ডাউনলোড করুন (Optional)

যদি আপনি মডেল লোকাল মেশিনে ডাউনলোড করতে চান:

In [ ]:
# মডেল ZIP করুন
!cd /content/drive/MyDrive && zip -r qwen_nctb_final.zip qwen_nctb_final/

print("✅ মডেল ZIP করা হয়েছে!")
print("\n📁 ডাউনলোড লোকেশন:")
print("  /content/drive/MyDrive/qwen_nctb_final.zip")
print("\n💡 Google Drive থেকে এই ফাইল ডাউনলোড করুন।")

## 📊 ধাপ ১৪: Training মেট্রিক্স দেখুন

In [ ]:
# Training history দেখুন
import pandas as pd
import matplotlib.pyplot as plt

# Training logs থেকে loss extract করুন
logs = trainer.state.log_history

train_loss = [log['loss'] for log in logs if 'loss' in log]
eval_loss = [log['eval_loss'] for log in logs if 'eval_loss' in log]

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_loss, label='Training Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
if eval_loss:
    plt.plot(eval_loss, label='Eval Loss', color='orange')
    plt.xlabel('Eval Steps')
    plt.ylabel('Loss')
    plt.title('Evaluation Loss Over Time')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\n📊 Final Training Loss: {train_loss[-1]:.4f}")
if eval_loss:
    print(f"📊 Final Eval Loss: {eval_loss[-1]:.4f}")

## 🎉 সম্পন্ন!

অভিনন্দন! আপনি সফলভাবে **Qwen2.5-0.5B** মডেলকে বাংলা NCTB ডেটাসেট দিয়ে ফাইন-টিউন করেছেন।

### 📁 আপনার ফাইলসমূহ:

1. **ট্রেন করা মডেল**: `/content/drive/MyDrive/qwen_nctb_final/`
2. **চেকপয়েন্টস**: `/content/drive/MyDrive/qwen_nctb_finetuned/`
3. **ZIP ফাইল** (যদি তৈরি করে থাকেন): `/content/drive/MyDrive/qwen_nctb_final.zip`

### 🚀 পরবর্তী ধাপ:

1. **আরও ডেটা যোগ করুন**: আরও NCTB প্রশ্ন-উত্তর যোগ করে পুনরায় ট্রেন করুন
2. **মডেল অপটিমাইজ করুন**: হাইপারপ্যারামিটার টিউন করুন
3. **Android-এ ডিপ্লয় করুন**: ONNX বা TFLite-এ কনভার্ট করুন
4. **ওয়েব অ্যাপ তৈরি করুন**: Gradio বা Streamlit দিয়ে ইন্টারফেস বানান

---

**প্রশ্ন বা সমস্যা থাকলে GitHub-এ ইস্যু করুন!**

**তৈরি করেছেন**: Kiro AI 🤖